# 06 - Document Transformers Test
Testing document-level transformers: JSONToRowsTransformer, TextChunkerTransformer, PIIRedactorProvider

In [ ]:
!pip install -q sanitary


In [ ]:
from document_graph.transform.transformer_provider_config import TransformerProviderConfig
from document_graph.transform.document_transformers.json_to_rows import JSONToRowsTransformer
from document_graph.transform.document_transformers.text_chunker import TextChunkerTransformer
from document_graph.transform.document_transformers.pii_redactor_provider import PIIRedactorProvider

## JSONToRowsTransformer

In [ ]:
records = [
    {
        'source': 'vulnerability_scan',
        'content': '[{"cve": "CVE-2026-0001", "severity": "Critical", "asset": "web-server-01"}, {"cve": "CVE-2026-0002", "severity": "High", "asset": "db-server-03"}]'
    }
]
config = TransformerProviderConfig(name='json_to_rows', args={'json_field': 'content'})
transformer = JSONToRowsTransformer(config)
result = transformer.transform(records)
print("JSONToRowsTransformer:")
print(f"Input: 1 record -> Output: {len(result)} records")
for r in result:
    print(f"  row_index={r.get('row_index')}: cve={r.get('json_cve')}, severity={r.get('json_severity')}, asset={r.get('json_asset')}")

## TextChunkerTransformer

In [ ]:
long_policy = """Information Security Policy: All employees must follow secure access procedures. 
Multi-factor authentication is required for all production systems. Passwords must be rotated 
every 90 days and meet complexity requirements of at least 12 characters with mixed case, 
numbers and special characters. Access reviews must be conducted quarterly for all privileged 
accounts. Service accounts must not have interactive login capabilities. All access requests 
must be approved by the asset owner and the security team before provisioning. Terminated 
employees must have access revoked within 24 hours of separation."""

records = [{'id': 'POL-SEC-001', 'title': 'Access Control Policy', 'content': long_policy}]
config = TransformerProviderConfig(name='chunk_policy', args={
    'chunk_size': 150,
    'overlap': 20,
    'text_field': 'content'
})
transformer = TextChunkerTransformer(config)
result = transformer.transform(records)
print("TextChunkerTransformer:")
print(f"Input: 1 record ({len(long_policy)} chars) -> Output: {len(result)} chunks")
for r in result:
    print(f"  chunk {r['chunk_index']}: [{r['chunk_start']}:{r['chunk_end']}] '{r['content'][:50]}...'")

## PIIRedactorProvider

In [ ]:
records = [
    {
        'incident_id': 'INC-001',
        'description': 'User john.doe@company.com logged in from 192.168.1.100 with password Admin123!',
        'analyst_notes': 'Contacted admin@internal.corp at 10.0.0.55 for verification'
    },
    {
        'incident_id': 'INC-002',
        'description': 'SSH key exposed by user jane.smith at 172.16.0.42',
        'analyst_notes': 'Key rotated for service account svc-deploy'
    }
]
config = TransformerProviderConfig(name='redact_pii', args={
    'fields': ['description', 'analyst_notes']
})
transformer = PIIRedactorProvider(config)
result = transformer.transform(records)
print("PIIRedactorProvider:")
for r in result:
    print(f"  {r['incident_id']}:")
    print(f"    description: {r['description']}")
    print(f"    notes: {r['analyst_notes']}")

## Summary
All document transformers tested successfully:
- **JSONToRowsTransformer**: Flatten nested JSON into tabular rows with json_ prefix
- **TextChunkerTransformer**: Split long text into overlapping chunks with metadata
- **PIIRedactorProvider**: Redact PII (emails, IPs, credentials) from specified fields